## 03 — Plate-solve result display

Plate-solves an image and displays the result with:
- WCS sky-coordinate grid
- **White circles** — all sources submitted to astrometry.net
- **Green circles** — sources confirmed as catalog matches

In [ ]:
import sys
import os
from pathlib import Path
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

sys.path.insert(0, os.path.abspath('..'))

from extractor.platesolve import platesolve
from astropy.io import fits as afits
from astropy.wcs import WCS

data_dir = Path('../data')
fits_files = sorted(data_dir.glob('*.fit')) + sorted(data_dir.glob('*.fits'))
if not fits_files:
    raise FileNotFoundError(f"No FITS files found in {data_dir.resolve()}")

image_path = fits_files[0]
print(f"Using: {image_path}")

with afits.open(image_path) as hdul:
    image = hdul[0].data.astype(float)

In [ ]:
result = platesolve(image_path, write=False, verbose=True)

In [ ]:
if result is None:
    print("Solve failed.")
else:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        wcs = WCS(result.header)

    # Arcsinh stretch: clips at 0.5 / 99.5 percentile, then arcsinh-scales
    # so bright stars don't dominate while faint structure stays visible.
    finite = image[np.isfinite(image)]
    lo, hi = np.percentile(finite, [0.5, 99.5])
    disp = np.arcsinh(np.clip(image, lo, hi))

    fig, ax = plt.subplots(figsize=(12, 9), subplot_kw={'projection': wcs})
    ax.imshow(disp, origin='lower', cmap='gray',
              vmin=np.arcsinh(lo), vmax=np.arcsinh(hi))

    # WCS grid
    ax.coords.grid(True, color='cyan', alpha=0.35, linestyle='--', linewidth=0.7)
    ax.coords['ra'].set_axislabel('RA')
    ax.coords['dec'].set_axislabel('Dec')

    # All submitted sources — white
    for x, y in zip(result.detected_x, result.detected_y):
        ax.add_patch(Circle((x, y), radius=12,
                             edgecolor='white', facecolor='none',
                             linewidth=0.7, alpha=0.7))

    # Astrometry-confirmed matches — green
    for x, y in zip(result.matched_x, result.matched_y):
        ax.add_patch(Circle((x, y), radius=12,
                             edgecolor='lime', facecolor='none',
                             linewidth=1.2, alpha=0.9))

    ra  = result.header.get('CRVAL1', float('nan'))
    dec = result.header.get('CRVAL2', float('nan'))
    ax.set_title(
        f"{image_path.name}   RA={ra:.4f}°  Dec={dec:.4f}°  "
        f"detected={len(result.detected_x)}  matched={len(result.matched_x)}",
        fontsize=10,
    )

    # Restore pixel limits so circles aren't clipped
    ax.set_xlim(-0.5, image.shape[1] - 0.5)
    ax.set_ylim(-0.5, image.shape[0] - 0.5)

    plt.tight_layout()
    plt.show()